**โจทย์:** ประเมินความเข้มกลิ่นจากค่าของเซนเซอร์ เพื่อตรวจจับเหตุการณ์ผิดปกติ (Anomaly Detection)  
**กระบวนการ:** Data Science Process 6 ขั้นตอน

---
## Step 1 — Setting the Research Goal

### บริบทของปัญหา
หน้างาน (โรงงาน/โรงบำบัดน้ำเสีย) ติดตั้งสถานีตรวจวัดกลิ่นต่อเนื่อง ชุมชนรอบข้างร้องเรียนเรื่องกลิ่นรบกวนบ่อยขึ้น ผู้จัดการต้องการเครื่องมือที่ช่วยให้:
- เข้าใจสถานการณ์กลิ่นจากข้อมูล sensor
- ตอบข้อร้องเรียนได้อย่างมีหลักฐาน
- ตัดสินใจได้เร็วขึ้นว่าควรแก้ไขที่จุดใด เมื่อใด

### Research Goal
> **"ประเมินความเข้มกลิ่น (D/T) จากค่าของ sensor ทั้ง 8 ตัว และตรวจจับช่วงเวลาที่กลิ่นผิดปกติ เพื่อสนับสนุนการตัดสินใจของหน้างาน"**

---
## Step 2 — Retrieving Data

ข้อมูลมาจากสถานีตรวจวัดกลิ่น ENose บันทึกค่าทุก **1 นาที** ต่อเนื่องประมาณ **91 วัน**  
ไฟล์: `Export.csv` — เป็นข้อมูลจริงที่ยังไม่ผ่านการทำความสะอาด

### หมายเหตุการโหลดไฟล์
ไฟล์นี้มีบรรทัดแรกเป็น `sep=,` ซึ่งเป็น Excel dialect — ต้องใช้ `skiprows=1` เพื่อข้ามบรรทัดนั้น

In [1]:
import pandas as pd
import numpy as np

# โหลดข้อมูล — skiprows=1 เพื่อข้ามบรรทัด sep=,
df = pd.read_csv("Export.csv", skiprows=1)

# แปลง Time เป็น datetime
df["Time"] = pd.to_datetime(df["Time"])

print(f"โหลดสำเร็จ: {df.shape[0]:,} แถว, {df.shape[1]} คอลัมน์")
print(f"ช่วงเวลา: {df['Time'].min()} ถึง {df['Time'].max()}")

โหลดสำเร็จ: 130,343 แถว, 17 คอลัมน์
ช่วงเวลา: 2026-02-12 00:00:00 ถึง 2026-05-14 12:31:00


### 2.1 ดูโครงสร้างข้อมูล

In [2]:
# ดูตัวอย่างข้อมูล
df.head(5)

,Time,D/T,Wind Direction,Wind Speed,Temperature,Relative Humidity,PM 2.5,Atmospheric Pressure,Sensor 1,Sensor 2,Sensor 3,Sensor 4,Sensor 5,Sensor 6,Sensor 7,Sensor 8,Smell Prediction
0,2026-02-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
1,2026-02-12 00:01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
2,2026-02-12 00:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
3,2026-02-12 00:03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient
4,2026-02-12 00:04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ambient


In [3]:
# ดู data type ของแต่ละคอลัมน์
df.dtypes

Time                    datetime64[ns]
D/T                            float64
Wind Direction                 float64
Wind Speed                     float64
Temperature                    float64
Relative Humidity              float64
PM 2.5                         float64
Atmospheric Pressure           float64
Sensor 1                       float64
Sensor 2                       float64
Sensor 3                       float64
Sensor 4                       float64
Sensor 5                       float64
Sensor 6                       float64
Sensor 7                       float64
Sensor 8                       float64
Smell Prediction                object
dtype: object

### 2.2 ตรวจสอบ Missing Values

In [2]:
# นับ missing values แต่ละคอลัมน์
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})
missing_summary[missing_summary["Missing Count"] > 0]

,Missing Count,Missing %
D/T,421,0.32
Wind Direction,422,0.32
Wind Speed,422,0.32
Temperature,422,0.32
Relative Humidity,422,0.32
PM 2.5,422,0.32
Atmospheric Pressure,422,0.32
Sensor 1,421,0.32
Sensor 2,421,0.32
Sensor 3,421,0.32


### 2.3 ดู Smell Prediction — target variable

In [3]:
# นับค่าใน Smell Prediction (รวม NaN)
print("=== Smell Prediction value_counts ===")
print(df["Smell Prediction"].value_counts(dropna=False))

print("\n=== แถวที่ Sensor ว่างทุกตัว ===")
sensor_cols = [f"Sensor {i}" for i in range(1, 9)]
all_sensor_null = df[sensor_cols].isnull().all(axis=1)
print(f"Sensor ว่างทุกตัว : {all_sensor_null.sum():,} แถว")
print(f"Sensor มีข้อมูล   : {(~all_sensor_null).sum():,} แถว")

=== Smell Prediction value_counts ===
Smell Prediction
Ambient        85504
NaN            44838
hackathon#2        1
Name: count, dtype: int64

=== แถวที่ Sensor ว่างทุกตัว ===
Sensor ว่างทุกตัว : 421 แถว
Sensor มีข้อมูล   : 129,922 แถว


### 2.4 สรุปสิ่งที่พบ

| ประเด็น | รายละเอียด | แนวทาง Step 3 |
|---|---|---|
| Sensor NaN 421 แถว | ~0.3% — sensor ว่างทุกตัว | ตัดทิ้ง |
| `hackathon#2` 1 แถว | ไม่ใช่ข้อมูลจริง | ตัดทิ้ง |
| Smell Prediction NaN 44,838 แถว | 34% — ยังไม่ทราบสาเหตุ | ต้องวิเคราะห์เพิ่ม |
| มีแค่ "Ambient" | ทั้งที่สเปกรองรับ 5 โปรไฟล์ | ต้องใช้ D/T แทน label |

---
## Step 3 — Data Preparation

ขั้นตอนนี้เป็นการทำความสะอาดและเตรียมข้อมูลจาก `Export.csv` ที่โหลดมาใน Step 2  
เพื่อให้ข้อมูลพร้อมสำหรับการทำ Data Exploration และ Data Modeling

จาก Step 2 พบปัญหาหลักคือ:
- มีแถวที่ Sensor 1–8 ว่างทุกตัว
- มี marker `hackathon#2` ซึ่งไม่ใช่ข้อมูลจริง
- `Smell Prediction` มีค่า NaN จำนวนมาก และมีเพียงค่า `Ambient`
- จึงเลือกใช้ `D/T` เป็น target หลักแทน `Smell Prediction`

### 3.1 กำหนดคอลัมน์ที่ใช้

In [ ]:
# กำหนดคอลัมน์ sensor
sensor_cols = [f"Sensor {i}" for i in range(1, 9)]

# คอลัมน์ตัวเลขที่ต้องใช้ในการวิเคราะห์
numeric_cols = [
    "D/T",
    "Wind Direction",
    "Wind Speed",
    "Temperature",
    "Relative Humidity",
    "PM 2.5",
    "Atmospheric Pressure"
] + sensor_cols

print("Sensor columns:", sensor_cols)
print("Numeric columns:", numeric_cols)

### 3.2 แปลงชนิดข้อมูลให้ถูกต้อง

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ถ้ายังไม่มี df ให้โหลดไฟล์ใหม่
if "df" not in globals():
    df = pd.read_csv("Export.csv", skiprows=1)

# ตัดช่องว่างชื่อคอลัมน์
df.columns = df.columns.str.strip()

sensor_cols = [f"Sensor {i}" for i in range(1, 9)]

numeric_cols = [
    "D/T",
    "Wind Direction",
    "Wind Speed",
    "Temperature",
    "Relative Humidity",
    "PM 2.5",
    "Atmospheric Pressure"
] + sensor_cols

# แปลง Time
df["Time"] = pd.to_datetime(df["Time"], errors="coerce")

# แปลงตัวเลข
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("✅ แปลงชนิดข้อมูลเรียบร้อย")
print("ขนาดข้อมูล:", df.shape)

display(df[numeric_cols + ["Time"]].dtypes.to_frame("dtype"))
display(df.head())

### 3.3 ลบข้อมูลที่ไม่ใช่ข้อมูลจริง

In [ ]:
before_rows = len(df)

df_clean = df[
    ~df["Smell Prediction"].astype(str).str.contains("hackathon", case=False, na=False)
].copy()

after_rows = len(df_clean)

print("✅ ลบข้อมูล marker ที่ไม่ใช่ข้อมูลจริง")
print(f"ก่อนลบ : {before_rows:,} แถว")
print(f"หลังลบ : {after_rows:,} แถว")
print(f"ลบออก : {before_rows - after_rows:,} แถว")

display(df_clean.head())

### 3.4 ลบแถวที่ Sensor 1–8 ว่างทั้งหมด

In [ ]:
before_rows = len(df_clean)

all_sensor_null = df_clean[sensor_cols].isna().all(axis=1)
removed_rows = df_clean[all_sensor_null]

df_clean = df_clean[~all_sensor_null].copy()

after_rows = len(df_clean)

print("✅ ลบแถวที่ Sensor 1–8 ว่างทั้งหมด")
print(f"ก่อนลบ : {before_rows:,} แถว")
print(f"หลังลบ : {after_rows:,} แถว")
print(f"ลบออก : {before_rows - after_rows:,} แถว")

print("\nตัวอย่างแถวที่ถูกลบ:")
display(removed_rows.head())

### 3.5 ตรวจสอบค่าที่อยู่นอกช่วงตามสเปก

In [ ]:
valid_ranges = {
    "D/T": (0, 60),
    "Wind Direction": (0, 360),
    "Wind Speed": (0, 40),
    "Temperature": (-40, 85),
    "Relative Humidity": (0, 100),
    "PM 2.5": (0, 1000),
    "Atmospheric Pressure": (500, 1100)
}

out_summary = []

for col, (low, high) in valid_ranges.items():
    out_of_range = df_clean[col].notna() & ~df_clean[col].between(low, high)
    out_summary.append({
        "column": col,
        "valid_range": f"{low} ถึง {high}",
        "out_of_range_rows": int(out_of_range.sum())
    })

out_summary_df = pd.DataFrame(out_summary)

print("✅ ตรวจสอบค่าที่อยู่นอกช่วงตามสเปก")
display(out_summary_df)

### 3.6 ลบแถวที่มีค่าหลุดช่วงตามสเปก

In [ ]:
before_rows = len(df_clean)

valid_mask = pd.Series(True, index=df_clean.index)

for col, (low, high) in valid_ranges.items():
    valid_mask = valid_mask & (
        df_clean[col].isna() | df_clean[col].between(low, high)
    )

removed_out_range = df_clean[~valid_mask]
df_clean = df_clean[valid_mask].copy()

after_rows = len(df_clean)

print("✅ ลบแถวที่มีค่าหลุดช่วงตามสเปก")
print(f"ก่อนลบ : {before_rows:,} แถว")
print(f"หลังลบ : {after_rows:,} แถว")
print(f"ลบออก : {before_rows - after_rows:,} แถว")

print("\nตัวอย่างแถวที่ถูกลบ:")
display(removed_out_range.head())

### 3.7 จัดการ Missing Values สำหรับคอลัมน์ที่ใช้ทำโมเดล

In [ ]:
feature_cols = sensor_cols + [
    "Wind Direction",
    "Wind Speed",
    "Temperature",
    "Relative Humidity",
    "PM 2.5",
    "Atmospheric Pressure"
]

target_col = "D/T"

before_rows = len(df_clean)

missing_before = df_clean[feature_cols + [target_col]].isna().sum().to_frame("missing_before")

df_clean = df_clean.dropna(subset=feature_cols + [target_col]).copy()

after_rows = len(df_clean)

missing_after = df_clean[feature_cols + [target_col]].isna().sum().to_frame("missing_after")
missing_summary = missing_before.join(missing_after)

print("✅ จัดการ Missing Values ใน feature และ target")
print(f"ก่อนลบ : {before_rows:,} แถว")
print(f"หลังลบ : {after_rows:,} แถว")
print(f"ลบออก : {before_rows - after_rows:,} แถว")

display(missing_summary)

### 3.8 สร้าง Feature จากเวลา

In [ ]:
df_clean["hour"] = df_clean["Time"].dt.hour
df_clean["day"] = df_clean["Time"].dt.day
df_clean["month"] = df_clean["Time"].dt.month
df_clean["dayofweek"] = df_clean["Time"].dt.dayofweek
df_clean["is_weekend"] = df_clean["dayofweek"].isin([5, 6]).astype(int)

time_cols = ["hour", "day", "month", "dayofweek", "is_weekend"]

print("✅ สร้าง Feature จากเวลาเรียบร้อย")
display(df_clean[["Time"] + time_cols].head())

### 3.9 เตรียมข้อมูลสำหรับขั้นตอน Modeling

In [ ]:
final_feature_cols = feature_cols + time_cols

X = df_clean[final_feature_cols]
y = df_clean[target_col]

print("✅ เตรียมข้อมูลสำหรับ Modeling เรียบร้อย")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

print("\nตัวอย่าง X:")
display(X.head())

print("\nตัวอย่าง y:")
display(y.head().to_frame("D/T"))

### 3.10 สรุปผลหลัง Data Preparation

In [ ]:
summary_data = {
    "รายการ": [
        "จำนวนข้อมูลหลังทำความสะอาด",
        "จำนวนคอลัมน์หลังทำความสะอาด",
        "เวลาเริ่มต้น",
        "เวลาสิ้นสุด",
        "จำนวน feature ที่ใช้ทำโมเดล",
        "target"
    ],
    "ค่า": [
        f"{df_clean.shape[0]:,} แถว",
        f"{df_clean.shape[1]} คอลัมน์",
        str(df_clean["Time"].min()),
        str(df_clean["Time"].max()),
        f"{len(final_feature_cols)} features",
        target_col
    ]
}

summary_df = pd.DataFrame(summary_data)

print("✅ Data Preparation Summary")
display(summary_df)

print("\nสถิติของ D/T หลังทำความสะอาด:")
display(df_clean["D/T"].describe().to_frame("D/T"))

### สรุป Step 3

ในขั้นตอน Data Preparation เราได้ทำความสะอาดข้อมูลจากไฟล์ `Export.csv` โดยลบข้อมูลที่ไม่ใช่ข้อมูลจริง เช่น `hackathon#2` ลบแถวที่ Sensor 1–8 ว่างทั้งหมด ตรวจสอบค่าที่อยู่นอกช่วงตามสเปก จัดการ missing values และสร้าง feature จากเวลา

หลังจากขั้นตอนนี้ ข้อมูลจะพร้อมสำหรับ Step 4 — Data Exploration และ Step 5 — Data Modeling โดยใช้ค่า `D/T` เป็น target สำหรับการประเมินความเข้มกลิ่น 